In [10]:
import sqlite3
import pandas as pd
import numpy as np

# Create connection to SQLite in-memory database
conn = sqlite3.connect(':memory:')

# 1. CREATE TABLES (The TaskFlow Schema)
conn.execute("""
CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    signup_date DATE,
    acquisition_channel VARCHAR(50),
    country VARCHAR(50),
    company_size VARCHAR(20)
);
""")

conn.execute("""
CREATE TABLE user_events (
    event_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    event_name VARCHAR(50),
    event_timestamp TIMESTAMP,
    platform VARCHAR(20)
);
""")

# 2. POPULATE WITH REALISTIC DATA (1,000 User Onboarding Simulation)
np.random.seed(42)
user_ids = list(range(1, 1001))
channels = ['Google_Search', 'LinkedIn_Ads', 'ProductHunt', 'Organic']
countries = ['USA', 'UK', 'Canada', 'Germany']
sizes = ['1-10', '11-50', '50+']

# Generate Users
users_list = []
for uid in user_ids:
    date = pd.to_datetime('2026-01-01') + pd.to_timedelta(np.random.randint(0, 90), unit='D')
    users_list.append((uid, str(date.date()), np.random.choice(channels), np.random.choice(countries), np.random.choice(sizes)))

conn.executemany("INSERT INTO users VALUES (?, ?, ?, ?, ?);", users_list)

# Generate Funnel Events (Drop-off behavior)
events_list = []
eid = 1
for uid, sdate, chan, ctry, size in users_list:
    base_time = pd.to_datetime(sdate)
    # Step 1: Everyone who signs up completes signup
    events_list.append((eid, uid, 'signup_completed', str(base_time), 'Web')); eid += 1

    # Step 2: 75% create a workspace
    if np.random.rand() < 0.75:
        t2 = base_time + pd.to_timedelta(np.random.randint(5, 30), unit='m')
        events_list.append((eid, uid, 'workspace_created', str(t2), 'Web')); eid += 1

        # Step 3: 50% add their first project
        if np.random.rand() < 0.65:
            t3 = t2 + pd.to_timedelta(np.random.randint(10, 60), unit='m')
            events_list.append((eid, uid, 'first_project_added', str(t3), 'Web')); eid += 1

            # Step 4: The Critical Step - Teammate Invite (Only 35% do this!)
            if np.random.rand() < 0.35:
                t4 = t3 + pd.to_timedelta(np.random.randint(20, 120), unit='m')
                events_list.append((eid, uid, 'invited_teammate', str(t4), 'Web')); eid += 1

                # Step 5: 80% of those who invited a teammate complete a task!
                if np.random.rand() < 0.80:
                    t5 = t4 + pd.to_timedelta(np.random.randint(30, 180), unit='m')
                    events_list.append((eid, uid, 'first_task_completed', str(t5), 'Web')); eid += 1

conn.executemany("INSERT INTO user_events VALUES (?, ?, ?, ?, ?);", events_list)
conn.commit()

print("✅ TaskFlow Database Created Successfully!")
print(f"Total Users: {len(users_list)}")
print(f"Total Events Logged: {len(events_list)}")

✅ TaskFlow Database Created Successfully!
Total Users: 1000
Total Events Logged: 2576


In [11]:
query = """
SELECT *
FROM users
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)
print(df)

   user_id signup_date acquisition_channel  country company_size
0        1  2026-02-21       Google_Search   Canada          50+
1        2  2026-03-13       Google_Search      USA          50+
2        3  2026-03-24         ProductHunt   Canada          50+
3        4  2026-03-29       Google_Search  Germany          50+
4        5  2026-01-22       Google_Search       UK        11-50


In [12]:
query = """
SELECT *
FROM user_events
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)
print(df)

   event_id  user_id         event_name      event_timestamp platform
0         1        1   signup_completed  2026-02-21 00:00:00      Web
1         2        1  workspace_created  2026-02-21 00:29:00      Web
2         3        2   signup_completed  2026-03-13 00:00:00      Web
3         4        2  workspace_created  2026-03-13 00:25:00      Web
4         5        3   signup_completed  2026-03-24 00:00:00      Web


In [13]:
query = """
SELECT
    u.user_id,
    u.acquisition_channel,
    e.event_name
FROM users AS u
JOIN user_events AS e
  ON u.user_id = e.user_id
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)
print(df)

   user_id acquisition_channel         event_name
0        1       Google_Search   signup_completed
1        1       Google_Search  workspace_created
2        2       Google_Search   signup_completed
3        2       Google_Search  workspace_created
4        3         ProductHunt   signup_completed


In [14]:
query = """
SELECT
    u.acquisition_channel,
    COUNT(DISTINCT CASE WHEN e.event_name = 'signup_completed' THEN u.user_id END) AS step_1_signups,
    COUNT(DISTINCT CASE WHEN e.event_name = 'workspace_created' THEN u.user_id END) AS step_2_workspaces,
    COUNT(DISTINCT CASE WHEN e.event_name = 'first_project_added' THEN u.user_id END) AS step_3_projects,
    COUNT(DISTINCT CASE WHEN e.event_name = 'invited_teammate' THEN u.user_id END) AS step_4_teammates,
    COUNT(DISTINCT CASE WHEN e.event_name = 'first_task_completed' THEN u.user_id END) AS step_5_tasks
FROM users AS u
JOIN user_events AS e
  ON u.user_id = e.user_id
GROUP BY u.acquisition_channel;
"""
df = pd.read_sql_query(query, conn)
print(df)

  acquisition_channel  step_1_signups  step_2_workspaces  step_3_projects  \
0       Google_Search             267                211              141   
1        LinkedIn_Ads             262                194              132   
2             Organic             236                175              116   
3         ProductHunt             235                176              115   

   step_4_teammates  step_5_tasks  
0                46            37  
1                44            34  
2                39            36  
3                46            34  


In [15]:
query = """
WITH user_birth_month AS (
    SELECT
        user_id,
        MIN(strftime('%Y-%m-01', event_timestamp)) AS cohort_month
    FROM user_events
    GROUP BY user_id
)
SELECT *
FROM user_birth_month
LIMIT 5;
"""
df_birth = pd.read_sql_query(query, conn)
print(df_birth)

   user_id cohort_month
0        1   2026-02-01
1        2   2026-03-01
2        3   2026-03-01
3        4   2026-03-01
4        5   2026-01-01


In [16]:
query = """
WITH user_birth_month AS (
    SELECT
        user_id,
        MIN(strftime('%Y-%m-01', event_timestamp)) AS cohort_month
    FROM user_events
    GROUP BY user_id
),
user_activity_months AS (
    SELECT
        e.user_id,
        b.cohort_month,
        strftime('%Y-%m-01', e.event_timestamp) AS event_month,
        -- The Calendar Distance Formula:
        (CAST(strftime('%Y', e.event_timestamp) AS INT) - CAST(strftime('%Y', b.cohort_month) AS INT)) * 12 +
        (CAST(strftime('%m', e.event_timestamp) AS INT) - CAST(strftime('%m', b.cohort_month) AS INT)) AS month_number
    FROM user_events AS e
    JOIN user_birth_month AS b
      ON e.user_id = b.user_id
)
SELECT *
FROM user_activity_months
LIMIT 5;
"""
df_calendardistance = pd.read_sql_query(query, conn)
print(df_calendardistance)


   user_id cohort_month event_month  month_number
0        1   2026-02-01  2026-02-01             0
1        1   2026-02-01  2026-02-01             0
2        2   2026-03-01  2026-03-01             0
3        2   2026-03-01  2026-03-01             0
4        3   2026-03-01  2026-03-01             0


In [17]:
query = """
WITH user_birth_month AS (
    SELECT
        user_id,
        MIN(strftime('%Y-%m-01', event_timestamp)) AS cohort_month
    FROM user_events
    GROUP BY user_id
),
user_activity_months AS (
    SELECT
        e.user_id,
        b.cohort_month,
        (CAST(strftime('%Y', e.event_timestamp) AS INT) - CAST(strftime('%Y', b.cohort_month) AS INT)) * 12 +
        (CAST(strftime('%m', e.event_timestamp) AS INT) - CAST(strftime('%m', b.cohort_month) AS INT)) AS month_number
    FROM user_events AS e
    JOIN user_birth_month AS b
      ON e.user_id = b.user_id
)
-- STEP 3: GROUP & COUNT
SELECT
    cohort_month,
    month_number,
    COUNT(DISTINCT user_id) AS active_users
FROM user_activity_months
GROUP BY cohort_month, month_number
ORDER BY cohort_month ASC, month_number ASC;
"""
df_cohort = pd.read_sql_query(query, conn)
print(df_cohort)

  cohort_month  month_number  active_users
0   2026-01-01             0           358
1   2026-02-01             0           311
2   2026-03-01             0           331


In [19]:
conn.execute("""
CREATE TABLE IF NOT EXISTS subscriptions (
    subscription_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    plan_type VARCHAR(20),
    signup_date DATE
);
""")

# We will convert a portion of our users to paid subscriptions
np.random.seed(42)
paid_users = []
sub_id = 1
for uid in user_ids:
    # Users who invited a teammate have a higher chance of converting
    invited_teammate = any(e[1] == uid and e[2] == 'invited_teammate' for e in events_list)
    conversion_chance = 0.40 if invited_teammate else 0.05

    if np.random.rand() < conversion_chance:
        paid_users.append((sub_id, uid, 'Pro_Annual', '2026-04-01'))
        sub_id += 1

conn.executemany("INSERT OR REPLACE INTO subscriptions VALUES (?, ?, ?, ?);", paid_users)
conn.commit()
print(f"Created subscriptions table and populated {len(paid_users)} subscriptions.\n")

# 2. Run the original conversion analysis query now that the table exists
query = """
WITH user_flags AS (
    SELECT
        u.user_id,
        -- Did this user ever invite a teammate? (1 = Yes, 0 = No)
        MAX(CASE WHEN e.event_name = 'invited_teammate' THEN 1 ELSE 0 END) AS invited_team,
        -- Did this user convert to a paid subscription? (1 = Yes, 0 = No)
        MAX(CASE WHEN s.subscription_id IS NOT NULL THEN 1 ELSE 0 END) AS converted_to_paid
    FROM users AS u
    LEFT JOIN user_events AS e ON u.user_id = e.user_id
    LEFT JOIN subscriptions AS s ON u.user_id = s.user_id
    GROUP BY u.user_id
)
SELECT
    invited_team,
    COUNT(user_id) AS total_users,
    SUM(converted_to_paid) AS total_conversions,
    ROUND((SUM(converted_to_paid) * 100.0 / COUNT(user_id)), 2) AS conversion_rate_pct
FROM user_flags
GROUP BY invited_team;
"""

df_aha = pd.read_sql_query(query, conn)
print(df_aha)

Created subscriptions table and populated 124 subscriptions.

   invited_team  total_users  total_conversions  conversion_rate_pct
0             0          825                 44                 5.33
1             1          175                 80                45.71
